# 02 — XGBoost and LightGBM

**Overview:** fit dedicated XGBoost and LightGBM classifiers on the same shared dataset from
`01_classification_fundamentals.ipynb` and compare accuracy/training time against sklearn's
`GradientBoostingClassifier`.

**Required libraries:** `scikit-learn`, `xgboost`, `lightgbm`.

```
!pip install -q xgboost lightgbm
```

⚠️ **Environment note:** both libraries depend on an OpenMP runtime (`libomp` on macOS), which
may not be present on every machine. This notebook detects that at import time and **skips
gracefully with a clear message** instead of failing, so it always runs end to end — Colab
ships OpenMP already, so both libraries work there out of the box.

Connects to: `01_classification_fundamentals.ipynb`. Leads to: `12_ml_evaluation`.

### Abbreviations used in this notebook

| Term | Full form | What it is |
|---|---|---|
| **XGBoost** | eXtreme Gradient Boosting | Fast, regularised gradient-boosted decision-tree library. |
| **LightGBM** | Light Gradient Boosting Machine | Histogram-based gradient-boosted tree library, optimised for speed and memory. |
| **OpenMP** | Open Multi-Processing | Multi-threading runtime used by XGBoost/LightGBM; must be installed on macOS. |
| **F1** | F1 score | Harmonic mean of precision and recall. |
| **ROC** | Receiver Operating Characteristic | Curve of true-positive rate vs false-positive rate across thresholds. |
| **AUC** | Area Under the Curve | Single-number summary of a curve; ROC-AUC = chance a positive outranks a negative. |

## 0. Environment & shared dataset (same generator as notebook 1)

In [1]:
import time
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score

rng_seed = 42
X, y = make_classification(n_samples=500, n_features=10, n_informative=6,
                            n_redundant=2, n_classes=2, random_state=rng_seed)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=rng_seed, stratify=y)
print("train:", X_train.shape, "test:", X_test.shape)


train: (375, 10) test: (125, 10)


## 1. Baseline: sklearn's GradientBoostingClassifier (always available)

In [2]:
start = time.perf_counter()
gbc = GradientBoostingClassifier(random_state=rng_seed).fit(X_train, y_train)
gbc_time = time.perf_counter() - start
gbc_acc = accuracy_score(y_test, gbc.predict(X_test))
print(f"sklearn GradientBoostingClassifier: accuracy={gbc_acc:.3f}, train_time={gbc_time*1000:.1f}ms")

results = [("sklearn GradientBoosting", gbc_acc, gbc_time)]


sklearn GradientBoostingClassifier: accuracy=0.872, train_time=56.2ms


## 2. XGBoost (depth-wise growth, strong regularization controls)

💡 **Intuition:** XGBoost grows trees level-by-level (depth-wise) and exposes extensive
regularization hyperparameters (`reg_alpha`, `reg_lambda`, `gamma`) for controlling overfitting
directly.

In [3]:
try:
    import xgboost as xgb
    start = time.perf_counter()
    xgb_model = xgb.XGBClassifier(
        n_estimators=100, max_depth=4, learning_rate=0.1,
        eval_metric="logloss", random_state=rng_seed,
    ).fit(X_train, y_train)
    xgb_time = time.perf_counter() - start
    xgb_acc = accuracy_score(y_test, xgb_model.predict(X_test))
    print(f"XGBoost: accuracy={xgb_acc:.3f}, train_time={xgb_time*1000:.1f}ms")
    results.append(("XGBoost", xgb_acc, xgb_time))
except Exception as e:
    # Broad on purpose: xgboost raises its own XGBoostError (not ImportError/OSError) when its
    # native library fails to load (e.g. missing libomp on macOS), so a narrower except would
    # miss exactly the failure mode this cell needs to degrade gracefully from.
    print(f"Skipping XGBoost in this environment (not installed or a runtime library is "
          f"missing, e.g. libomp on macOS): {type(e).__name__}: {e}")
    print("On Colab, or after `brew install libomp` + `pip install xgboost` on macOS, this cell "
          "would run normally.")


Skipping XGBoost in this environment (not installed or a runtime library is missing, e.g. libomp on macOS): XGBoostError: 
XGBoost Library (libxgboost.dylib) could not be loaded.
Likely causes:
  * OpenMP runtime is not installed
    - vcomp140.dll or libgomp-1.dll for Windows
    - libomp.dylib for Mac OSX
    - libgomp.so for Linux and other UNIX-like OSes
    Mac OSX users: Run `brew install libomp` to install OpenMP runtime.

  * You are running 32-bit Python on a 64-bit OS

Error message(s): ["dlopen(/Users/USER/aiprojects/llm-fine-tuning/.venv312/lib/python3.12/site-packages/xgboost/lib/libxgboost.dylib, 0x0006): Library not loaded: @rpath/libomp.dylib\n  Referenced from: <09B43001-8527-34F6-AAE4-B97DA8A6CBC0> /Users/USER/aiprojects/llm-fine-tuning/.venv312/lib/python3.12/site-packages/xgboost/lib/libxgboost.dylib\n  Reason: tried: '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such f

## 3. LightGBM (leaf-wise growth, histogram-based splits)

💡 **Intuition:** LightGBM grows the single leaf that most reduces loss next (leaf-wise, not
level-by-level) and bins continuous features into histograms before splitting — both choices
trade a bit of accuracy variance for a large speed advantage on big datasets.

In [4]:
try:
    import lightgbm as lgb
    start = time.perf_counter()
    lgb_model = lgb.LGBMClassifier(
        n_estimators=100, max_depth=4, learning_rate=0.1,
        random_state=rng_seed, verbosity=-1,
    ).fit(X_train, y_train)
    lgb_time = time.perf_counter() - start
    lgb_acc = accuracy_score(y_test, lgb_model.predict(X_test))
    print(f"LightGBM: accuracy={lgb_acc:.3f}, train_time={lgb_time*1000:.1f}ms")
    results.append(("LightGBM", lgb_acc, lgb_time))
except Exception as e:
    # Broad for the same reason as the XGBoost cell above: the native-library load failure
    # doesn't surface as a plain ImportError/OSError.
    print(f"Skipping LightGBM in this environment (not installed or a runtime library is "
          f"missing, e.g. libomp on macOS): {type(e).__name__}: {e}")
    print("On Colab, or after `brew install libomp` + `pip install lightgbm` on macOS, this "
          "cell would run normally.")


Skipping LightGBM in this environment (not installed or a runtime library is missing, e.g. libomp on macOS): OSError: dlopen(/Users/USER/aiprojects/llm-fine-tuning/.venv312/lib/python3.12/site-packages/lightgbm/lib/lib_lightgbm.dylib, 0x0006): Library not loaded: @rpath/libomp.dylib
  Referenced from: <59CCDAC2-EBAE-32F7-ADF4-469B834532C9> /Users/USER/aiprojects/llm-fine-tuning/.venv312/lib/python3.12/site-packages/lightgbm/lib/lib_lightgbm.dylib
  Reason: tried: '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/opt/local/lib/libomp/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/local/lib/libomp/libomp.dylib' (no such file), '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/opt/local/lib/libomp/libomp.dylib' (no such file), '/System/Volumes/Preboot/

## 4. Comparison

In [5]:
print(f"{'Model':30s} {'Accuracy':>10s} {'Train time (ms)':>18s}")
for name, acc, t in results:
    print(f"{name:30s} {acc:10.3f} {t*1000:18.1f}")

assert len(results) >= 1          # at minimum, the always-available sklearn baseline ran
assert all(acc > 0.5 for _, acc, _ in results)
print(f"\n{len(results)} of 3 possible gradient-boosting implementations ran in this environment.")


Model                            Accuracy    Train time (ms)
sklearn GradientBoosting            0.872               56.2

1 of 3 possible gradient-boosting implementations ran in this environment.


## 5. Connection to what's next

All the models across both classification notebooks have so far been judged on accuracy alone —
`12_ml_evaluation` introduces precision/recall/F1/ROC-AUC, which matter far more once classes
are imbalanced.

## 6. Interview questions
1. What's the structural difference between depth-wise (XGBoost) and leaf-wise (LightGBM) tree growth?
2. Why can leaf-wise growth be faster but also more prone to overfitting on small datasets?
3. What's the purpose of histogram-based binning for continuous features in LightGBM?
4. Why might XGBoost and sklearn's `GradientBoostingClassifier` produce similar but not identical results on the same data?
5. What's a reasonable way to design code that depends on an optional, environment-sensitive library like XGBoost?

## 7. Exercises
1. If XGBoost ran successfully, tune `max_depth` from 2 to 8 and observe the accuracy/overfitting tradeoff.
2. Compare training time scaling for sklearn's GradientBoosting vs LightGBM as `n_estimators` grows from 50 to 500.
3. Read XGBoost's feature importances (`xgb_model.feature_importances_`) and compare them to the dataset's known informative feature count.

## 8. Summary

✅ XGBoost and LightGBM are production-grade, highly-tunable gradient boosting implementations;
the core boosting idea is the same one demonstrated with sklearn's `GradientBoostingClassifier`
in `01_classification_fundamentals.ipynb`. This closes `08_classification` — next:
`09_unsupervised_learning`.